# Sensitivity Analysis — Fixed Encoding

The original `sensitivity_analysis.ipynb` produces a heatmap whose yellow plateau visually reads as `c* ≈ 15`, but that value is actually a **sentinel** injected by `extract_thresholds_by_t` whenever the converged value-iteration policy contains no `replace` action anywhere on the cum-context grid for a given $t$:

```python
if np.any(replace_mask):
    thresholds[t_idx] = float(dpagent.grids[0][np.argmax(replace_mask)])
else:
    thresholds[t_idx] = float(dpagent.grids[0][-1])  # never replace within grid
```

Empirically (verified by re-running with `max_cumulative_context = 30, 60` at the same 0.15 grid spacing), the yellow regions are **genuine "never replace"** — not censored thresholds. The dynamics saturate around $c_c \gtrsim 10$ (the survival probability is already $\approx 0$ there for any reasonable customer), so extending the cap doesn't reveal a finite threshold beyond 15. The yellow plateau is therefore a real subset of parameter space, and what the original plot does badly is *blend* it with the viridis colorbar so a reader reads `c* ≈ 15` off the legend.

This notebook reuses the saved `data/sensitivity_*.pkl` matrices (no value iteration is re-run) and produces two clearer plots:

1. **Interior threshold heatmap** — never-replace cells are masked to NaN and rendered in grey, so the smooth/monotonic interior dependence of $c^*$ on parameters is visible without a misleading plateau.
2. **Boundary indicator** — a binary partition between "interior threshold exists" and "never replace," exposing the shape and location of the never-replace region. Two distinct never-replace corners appear in the $F$–$\lambda$ panel: a *low-$F$* corner (replacement gain too small to recover $R$) and a *high-$F$, high-$\lambda$* corner (even fresh machines fail too fast to recover $R$). This dual structure is the structurally interesting observation worth a paragraph in the paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from itertools import combinations
import pickle
import os

## Load saved sensitivity matrices

We reuse the existing pickles in `../data/sensitivity_<p1>_<p2>.pkl`. Each contains `matrix`, `axis1_values`, `axis2_values`. The sentinel value used by `extract_thresholds_by_t` is `grids[0][-1] = max_cumulative_context = 15.0`.

In [ ]:
DATA_DIR = '../data'
FIG_DIR = '../figures'

# Default parameter values used in the main experiment
DEFAULTS = {'R': 1.5, 'F': 1.2, 'h': 0.02, 'lam': 0.001}

PARAM_LABELS = {
    'R':   r'Replacement Cost $R$',
    'F':   r'Failure Cost $F$',
    'h':   r'Holding Cost Rate $h$',
    'lam': r'Baseline Hazard $\lambda$',
}

param_names = ['R', 'F', 'h', 'lam']
pairs = list(combinations(param_names, 2))

results = {}
ranges = {}
for p1, p2 in pairs:
    path = f'{DATA_DIR}/sensitivity_{p1}_{p2}.pkl'
    with open(path, 'rb') as f:
        d = pickle.load(f)
    results[(p1, p2)] = d['matrix']
    ranges[p1] = d['axis1_values']
    ranges[p2] = d['axis2_values']
    print(f'Loaded {p1} vs {p2}: matrix shape {d["matrix"].shape}')

# Sentinel value (grids[0][-1]) — must match max_cumulative_context in config.training_hyperparams
GRID_MAX_CC = 15.0
print(f'\nSentinel (never-replace marker) = {GRID_MAX_CC}')

## Step 1 — Mask never-replace cells

A cell hits exactly `GRID_MAX_CC` only when `extract_thresholds_by_t` found no `replace` action anywhere along the cum-context axis. Replacing those cells with `NaN` separates "censored / never replace" from the genuine interior values.

In [ ]:
def mask_never_replace(matrix, sentinel=GRID_MAX_CC, tol=1e-6):
    """Return a copy of `matrix` with sentinel cells replaced by NaN."""
    masked = matrix.astype(np.float64).copy()
    masked[np.isclose(matrix, sentinel, atol=tol)] = np.nan
    return masked

masked_results = {pair: mask_never_replace(m) for pair, m in results.items()}

for pair, m in masked_results.items():
    n_never = int(np.sum(np.isnan(m)))
    n_total = m.size
    interior_lo = np.nanmin(m) if n_never < n_total else np.nan
    interior_hi = np.nanmax(m) if n_never < n_total else np.nan
    print(f'{pair}: {n_never:3d}/{n_total} never-replace cells; '
          f'interior c* range = [{interior_lo:.3f}, {interior_hi:.3f}]')

## Step 2 — Interior threshold heatmap

Same six-panel layout as the original plot, but never-replace cells are rendered in grey via `cmap.set_bad(...)` and the colorbar is restricted to the interior range. Now the smooth, monotonic dependence of $c^*$ on $R$, $F$, $\lambda$ is visible without the misleading yellow plateau dominating the colorbar.

Saved to `figures/sensitivity_interior_threshold.pdf` (does **not** overwrite the original `sensitivity_departure_threshold.pdf`).

In [ ]:
def make_interior_cmap():
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad(color='lightgrey', alpha=1.0)
    return cmap

# Common color limits over the interior of all pairs (excludes NaN sentinel cells).
interior_vals = np.concatenate([m[~np.isnan(m)].ravel() for m in masked_results.values()
                                if np.any(~np.isnan(m))])
vmin = float(np.min(interior_vals))
vmax = float(np.max(interior_vals))
print(f'Shared color limits for interior c*: [{vmin:.3f}, {vmax:.3f}]')

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes_flat = axes.flatten()
cmap = make_interior_cmap()

for idx, (p1, p2) in enumerate(pairs):
    ax = axes_flat[idx]
    mat = masked_results[(p1, p2)]

    X, Y = np.meshgrid(ranges[p1], ranges[p2])
    pcm = ax.pcolormesh(X, Y, mat.T, cmap=cmap, shading='auto', vmin=vmin, vmax=vmax)

    ax.axvline(DEFAULTS[p1], color='red', linestyle='--', alpha=0.8, linewidth=1.5)
    ax.axhline(DEFAULTS[p2], color='red', linestyle='--', alpha=0.8, linewidth=1.5)
    ax.plot(DEFAULTS[p1], DEFAULTS[p2], 'r*', markersize=12)

    ax.set_xlabel(PARAM_LABELS[p1], fontsize=16)
    ax.set_ylabel(PARAM_LABELS[p2], fontsize=16)
    ax.tick_params(labelsize=12)

    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(r'$c^*$ (interior threshold)', fontsize=14)
    cbar.ax.tick_params(labelsize=11)

# Single legend entry for the never-replace mask
never_patch = mpatches.Patch(facecolor='lightgrey', edgecolor='black',
                              label='Never replace (no $c^*$ on grid)')
fig.legend(handles=[never_patch], loc='upper right', fontsize=14,
           bbox_to_anchor=(0.99, 0.99))

fig.suptitle(r'Interior departure threshold $c^*$ '
             r'(never-replace cells shown in grey)',
             fontsize=22, y=1.02)
plt.tight_layout()
os.makedirs(FIG_DIR, exist_ok=True)
plt.savefig(f'{FIG_DIR}/sensitivity_interior_threshold.pdf', bbox_inches='tight')
plt.show()

## Step 3 — Boundary indicator

Binary partition: which $(p_1, p_2)$ cells admit an interior $c^*$, and which fall in the never-replace regime. This is the direction the paper could lean into structurally — the never-replace region is a real qualitative feature of the optimal SMDP policy, with two distinct mechanisms producing it depending on which corner you approach.

Saved to `figures/sensitivity_never_replace_boundary.pdf`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes_flat = axes.flatten()

# 0 = interior threshold exists, 1 = never replace
binary_cmap = ListedColormap(['#ffffff', '#d62728'])

for idx, (p1, p2) in enumerate(pairs):
    ax = axes_flat[idx]
    mat = masked_results[(p1, p2)]
    boundary = np.isnan(mat).astype(float)

    X, Y = np.meshgrid(ranges[p1], ranges[p2])
    ax.pcolormesh(X, Y, boundary.T, cmap=binary_cmap, shading='auto',
                  vmin=0, vmax=1, edgecolors='lightgrey', linewidth=0.3)

    ax.axvline(DEFAULTS[p1], color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axhline(DEFAULTS[p2], color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.plot(DEFAULTS[p1], DEFAULTS[p2], 'k*', markersize=12)

    ax.set_xlabel(PARAM_LABELS[p1], fontsize=16)
    ax.set_ylabel(PARAM_LABELS[p2], fontsize=16)
    ax.tick_params(labelsize=12)

handles = [
    mpatches.Patch(facecolor='#ffffff', edgecolor='black',
                   label='Interior threshold exists'),
    mpatches.Patch(facecolor='#d62728', edgecolor='black',
                   label='Never replace'),
]
fig.legend(handles=handles, loc='upper right', fontsize=14,
           bbox_to_anchor=(0.99, 0.99))

fig.suptitle('Boundary between interior threshold and never-replace regimes',
             fontsize=22, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/sensitivity_never_replace_boundary.pdf', bbox_inches='tight')
plt.show()

## Step 4 — Focused view of the $F$–$\lambda$ panel

The $F$ vs $\lambda$ panel is the most informative one — it's where both never-replace mechanisms appear simultaneously. Side-by-side: interior threshold (left) and binary partition (right), to make the dual-corner structure unambiguous in the paper.

In [ ]:
fig, (ax_int, ax_bnd) = plt.subplots(1, 2, figsize=(15, 6))

p1, p2 = 'F', 'lam'
mat = masked_results[(p1, p2)]
X, Y = np.meshgrid(ranges[p1], ranges[p2])

# Left: interior threshold
pcm = ax_int.pcolormesh(X, Y, mat.T, cmap=make_interior_cmap(), shading='auto')
ax_int.axvline(DEFAULTS[p1], color='red', linestyle='--', alpha=0.8, linewidth=1.5)
ax_int.axhline(DEFAULTS[p2], color='red', linestyle='--', alpha=0.8, linewidth=1.5)
ax_int.plot(DEFAULTS[p1], DEFAULTS[p2], 'r*', markersize=14)
ax_int.set_xlabel(PARAM_LABELS[p1], fontsize=14)
ax_int.set_ylabel(PARAM_LABELS[p2], fontsize=14)
ax_int.set_title(r'Interior $c^*$ (grey = never replace)', fontsize=15)
fig.colorbar(pcm, ax=ax_int).set_label(r'$c^*$', fontsize=13)

# Right: binary partition
boundary = np.isnan(mat).astype(float)
ax_bnd.pcolormesh(X, Y, boundary.T, cmap=ListedColormap(['#ffffff', '#d62728']),
                  shading='auto', vmin=0, vmax=1, edgecolors='lightgrey', linewidth=0.3)
ax_bnd.axvline(DEFAULTS[p1], color='black', linestyle='--', alpha=0.5, linewidth=1)
ax_bnd.axhline(DEFAULTS[p2], color='black', linestyle='--', alpha=0.5, linewidth=1)
ax_bnd.plot(DEFAULTS[p1], DEFAULTS[p2], 'k*', markersize=14)
ax_bnd.set_xlabel(PARAM_LABELS[p1], fontsize=14)
ax_bnd.set_ylabel(PARAM_LABELS[p2], fontsize=14)
ax_bnd.set_title('Never-replace boundary', fontsize=15)
ax_bnd.legend(handles=[
    mpatches.Patch(facecolor='#ffffff', edgecolor='black', label='Interior $c^*$'),
    mpatches.Patch(facecolor='#d62728', edgecolor='black', label='Never replace'),
], loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/sensitivity_F_lam_focus.pdf', bbox_inches='tight')
plt.show()

## Discussion — two never-replace regimes

**Why does an interior $c^*$ exist or not?** The departure-state Q-values compare

$$
q_{\mathrm{replace}} = -h\,\bar{\tau} - R + \gamma\,V_{\mathrm{arrival}}(c_c{=}0,\,t{=}0),
\qquad
q_{\mathrm{no\text{-}replace}}(c_c) = -h\,\bar{\tau} + \gamma\,\mathbb{E}[V_{\mathrm{arrival}}(c_c, t)].
$$

An interior threshold $c^*$ exists iff $q_{\mathrm{replace}} \geq q_{\mathrm{no\text{-}replace}}(c_c)$ for at least one $c_c$. As parameters change, both Q-values shift; when the maximum of $q_{\mathrm{no\text{-}replace}}$ across $c_c$ rises above $q_{\mathrm{replace}}$ uniformly, the policy switches to *never replace* at a finite parameter value — a genuine phase transition, not a censoring artifact.

Two distinct mechanisms produce the never-replace regime, and they show up as two corners of the $F$–$\lambda$ partition:

1. **Low-$F$ corner** (left edge of the $F$–$\lambda$ panel). With low failure cost, the agent absorbs failures cheaply, so the value gap $V_{\mathrm{arrival}}(c_c{=}0) - V_{\mathrm{arrival}}(c_c{\to}\infty)$ is small. Replacing buys little improvement, and the fixed cost $R$ is no longer recovered.
2. **High-$F$, high-$\lambda$ corner** (top-right of the $F$–$\lambda$ panel). Even fresh machines fail almost immediately, so the expected number of paying customers per replacement cycle drops below the $R$ break-even. Replacing simply resets to a state that is itself nearly worthless.

Between these two corners is a crescent-shaped *interior* region where Proposition 1's control-limit policy applies and $c^*$ varies smoothly with parameters in the directions one would expect:

- $c^*$ rises with $R$ (replacing is more costly, so wait longer)
- $c^*$ falls with $F$ (failures more costly, replace earlier)
- $c^*$ falls with $\lambda$ (faster baseline degradation, replace earlier)
- $c^*$ is essentially insensitive to $h$ (holding cost affects both branches of the comparison roughly equally)

**On the cap.** Increasing `max_cumulative_context` does **not** shrink the never-replace region: the dynamics saturate near $c_c \approx 10$ regardless of cap (since $\exp(c_c+c_x)\Delta\Lambda$ is already enormous there), so adding grid points beyond just adds dead cells. We verified this by re-running the corners at `max_cc = 30, 60` with grid spacing held constant at 0.15 — the never-replace decision is unchanged. The only effect of changing `max_cc` without scaling $N_{c_c}$ proportionally is to coarsen the grid spacing and **distort** interior thresholds; so if `max_cc` is changed, $N_{c_c}$ should be scaled to keep spacing $\approx 0.15$, at proportional VI cost.

**Recommendation for the paper.** Use the *interior* heatmap (Step 2) for the smoothness story, and the *boundary* plot (Step 3 / Step 4) as a structural observation: an optimal control-limit policy degrades to a no-replace policy in two distinct economic limits, both of which are captured numerically without proof. No analytical work required — the existing pickled matrices already contain everything.